In [ ]:
#TOV Equations for Hadronic Matter
def TOV_hadronic(r, P, M):
    
    #Safety check 
    if P <= 0.0:
        return 0.0, 0.0
    
    den_en = den_en_hadronic(P) #MeV/fm^3 
    den_en_g = den_en * MeVfm3_to_1km2  # 1/km^2 
    P_g  = P  * MeVfm3_to_1km2  # 1/km^2 

    rr = max(r, 1e-12)  #Avoid division by zero in dP/dr
    
    dPdr_g = -(den_en_g + P_g) * (M + 4*np.pi*rr**3*P_g) / (rr*(rr - 2*M))
    dPdr   = dPdr_g / MeVfm3_to_1km2
    dmdr   = 4*np.pi*rr**2 * den_en_g
    return dPdr, dmdr

#TOV Equations for Hybrid Matter
def TOV_hybrid(r, P, M, den_en_function):
    
    #Safety check 
    if P <= 0.0:
        return 0.0, 0.0
        
    den_en = den_en_function(P) 
    den_en_g = den_en * MeVfm3_to_1km2
    P_g  = P  * MeVfm3_to_1km2
    
    rr = max(r, 1e-12)

    dPdr_g = -(den_en_g + P_g) * (M + 4*np.pi*rr**3*P_g) / (rr*(rr - 2*M))
    dPdr   = dPdr_g / MeVfm3_to_1km2
    dmdr   = 4*np.pi*rr**2 * den_en_g
    return dPdr, dmdr

#------------------------------------------------

#Numerical integration of TOV for Hadronic Matter (Runge–Kutta 4th order)
def integrate_star_hadronic(Pc, h0=0.01, P_R=1e-12, Rmax=100):
    #Pc: central pressure [MeV/fm3]
    #h0: first integration step [km]
    #P_R: pressure at surface (≈0)
    #R: maximum integration radius [km]
    
    r = float(h0)
    P = float(Pc)
    den_en_c = den_en_hadronic(Pc)
    den_en_c_g = den_en_c * MeVfm3_to_1km2
    m = (4/3) * np.pi * (r**3) * den_en_c_g

    r_list, P_list, m_list = [0.0, r], [Pc, P], [0.0, m]
    h = float(h0)
    
    #Runge–Kutta 4th order
    while P > P_R and r < Rmax:
        k1P, k1m = TOV_hadronic(r, P, m)
        k2P, k2m = TOV_hadronic(r + 0.5*h, P + 0.5*h*k1P, m + 0.5*h*k1m)
        k3P, k3m = TOV_hadronic(r + 0.5*h, P + 0.5*h*k2P, m + 0.5*h*k2m)
        k4P, k4m = TOV_hadronic(r + h,     P + h*k3P,     m + h*k3m)

        #Next step
        rn = r + h
        Pn = P + (h/6)*(k1P + 2*k2P + 2*k3P + k4P)
        mn = m + (h/6)*(k1m + 2*k2m + 2*k3m + k4m)

        if Pn < 0.0: 
            Pn = 0.0  #Safety check

        #Store next step
        r_list.append(rn) 
        P_list.append(Pn) 
        m_list.append(mn)
        r, P, m = rn, Pn, mn

        #Adaptive step size based on pressure scale height H = |P/dPdr|
        dPdr = TOV_hadronic(r, max(P, 1e-300), m)[0] #ΤΟV[0] for dPdr, max: avoid division by zero
        if dPdr != 0.0:
            H = abs(P/dPdr) #Scale height
            h = float(np.clip(0.05*H, 5e-3, 0.3))   #New step,limits: min,max

    #Linear Interpolation for surface (P=10^-12) - When P<0, linear interpolate to find zero point
    if len(P_list) >= 2 and P_list[-2] != P_list[-1]:
        P1, P2 = P_list[-1], P_list[-2]
        r1, r2 = r_list[-1], r_list[-2]
        m1, m2 = m_list[-1], m_list[-2]
        R_had = r1 + (P_R - P1)*(r2 - r1)/(P2 - P1)
        M_had = m1 + (P_R - P1)*(m2 - m1)/(P2 - P1)
    else:
        R_had, M_had = r_list[-1], m_list[-1]

    return R_had, M_had

#Numerical integration of TOV for Hybrid Matter (Runge–Kutta 4th order)
def integrate_star_hybrid(Pc, den_en_function, h0=0.01, P_R=1e-12, Rmax=100):
    #Pc: central pressure [MeV/fm3]
    #h0: first integration step [km]
    #P_R: pressure at surface (≈0)
    #R: maximum integration radius [km]
    
    r = float(h0)   #safety check 
    P = float(Pc)   #safety check 
    den_en_c = den_en_function(Pc)
    den_en_c_g  = den_en_c * MeVfm3_to_1km2
    m = (4 / 3) * np.pi * (r ** 3) * den_en_c_g

    r_list, P_list, m_list = [0.0, r], [Pc, P], [0.0, m]
    h = float(h0)

    #Runge–Kutta 4th order
    while P > P_R and r < Rmax:
        k1P, k1m = TOV_hybrid(r, P, m, den_en_function)
        k2P, k2m = TOV_hybrid(r + 0.5*h, P + 0.5*h*k1P, m + 0.5*h*k1m, den_en_function)
        k3P, k3m = TOV_hybrid(r + 0.5*h, P + 0.5*h*k2P, m + 0.5*h*k2m, den_en_function)
        k4P, k4m = TOV_hybrid(r + h, P + h*k3P, m + h*k3m, den_en_function)

        #Next step
        rn = r + h
        Pn = P + (h / 6) * (k1P + 2*k2P + 2*k3P + k4P)
        mn = m + (h / 6) * (k1m + 2*k2m + 2*k3m + k4m)

        if Pn < 0.0:
            Pn = 0.0  # Safety check

        #Store next step
        r_list.append(rn)
        P_list.append(Pn)
        m_list.append(mn)
        r, P, m = rn, Pn, mn

        #Adaptive step size based on pressure scale height H=|P/dPdr| 
        dPdr = TOV_hybrid(r, max(P, 1e-300), m, den_en_function)[0]  #ΤΟV[0] for dPdr, max: avoid division by zero
        if dPdr != 0.0:
            H = abs(P/dPdr) #Scale height 
            h = float(np.clip(0.05*H, 5e-3, 0.3))   #New step,limits: min,max 
    
    #Linear Interpolation for surface (P=10^-12) - When P<0, linear interpolate to find zero point
    if len(P_list) >= 2 and P_list[-2] != P_list[-1]:
        P1, P2 = P_list[-1], P_list[-2]
        r1, r2 = r_list[-1], r_list[-2]
        m1, m2 = m_list[-1], m_list[-2]
        R_hyb = r1 + (P_R - P1) * (r2 - r1) / (P2 - P1)
        M_hyb = m1 + (P_R - P1) * (m2 - m1) / (P2 - P1)   
    else:
        R_hyb, M_hyb = r_list[-1], m_list[-1]
        
    #Return final radius and mass of the star
    return R_hyb, M_hyb
